# Lab 17: Bank ATM System Simulation

**Problem Statement:**
Model an ATM vestibule where customers arrive at random, use one of several machines, and leave. Calculate the maximum queue lengths.


### Theory and Approach

This is a **Multi-Server Queue Simulation (M/M/c)**.
- **c**: Number of identical servers (ATMs).
- **Queue**: A single shared queue (FIFO). When an ATM becomes free, the first person in the queue goes to it.
- **Arrivals**: Customers arrive according to a Poisson process (Exponential inter-arrival times).
- **Service**: Usage time at the ATM follows an Exponential distribution.

We need to track the number of customers in the queue and record the maximum length observed during the simulation.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# ==========================================
# 1. Simulation Parameters
# ==========================================
NUM_ATMS = 3
SIM_HOURS = 10.0
SIM_TIME = SIM_HOURS * 60 # Simulating in minutes

MEAN_INTERARRIVAL = 2.0   # On average, a customer arrives every 2 mins
MEAN_SERVICE_TIME = 5.0   # On average, it takes 5 mins at the ATM

# State Variables
clock = 0.0
# List of ATMs indicating when they will be free. 0 means free now.
atms_free_time = [0.0] * NUM_ATMS 
queue = [] # Queue holds the arrival times of customers

# Tracking Variables
max_queue_length = 0
customers_served = 0

next_arrival = np.random.exponential(MEAN_INTERARRIVAL)

# For plotting
time_points = [0.0]
queue_lengths = [0]

print(f"--- Simulating {NUM_ATMS} ATMs for {SIM_HOURS} hours ---")

# ==========================================
# 2. Simulation Loop
# ==========================================
# To keep it simple, we process time event by event.
# Events are either: Next Arrival, or Next ATM becoming free
while clock < SIM_TIME:
    # Next departure is the minimum free time among busy ATMs
    # Only consider ATMs that are currently busy (free_time > clock)
    busy_atms = [t for t in atms_free_time if t > clock]
    next_departure = min(busy_atms) if busy_atms else float('inf')
    
    # Determine the next event
    next_event_time = min(next_arrival, next_departure)
    
    if next_event_time > SIM_TIME:
        break
        
    clock = next_event_time
    
    # --- PROCESS ARRIVAL ---
    if next_event_time == next_arrival:
        # Customer arrives
        queue.append(clock)
        
        # Update max queue length
        if len(queue) > max_queue_length:
            max_queue_length = len(queue)
            
        # Schedule next arrival
        next_arrival = clock + np.random.exponential(MEAN_INTERARRIVAL)
        
    # --- PROCESS DEPARTURE & ASSIGN IDLE ATMs ---
    # Assign customers in queue to any free ATMs
    # (This handles both the case where an ATM just became free, 
    # or an ATM was already free when a customer arrived)
    for i in range(NUM_ATMS):
        if atms_free_time[i] <= clock and len(queue) > 0:
            # ATM i is free and someone is in queue
            customer_arrival_time = queue.pop(0)
            
            # Schedule completion for this ATM
            service_time = np.random.exponential(MEAN_SERVICE_TIME)
            atms_free_time[i] = clock + service_time
            customers_served += 1
            
    # Record state for plotting
    time_points.append(clock)
    queue_lengths.append(len(queue))

# ==========================================
# 3. Results and Visualization
# ==========================================
print(f"Total Customers Served : {customers_served}")
print(f"Maximum Queue Length   : {max_queue_length} customers")

plt.figure(figsize=(14, 5))
plt.plot(time_points, queue_lengths, color='#17becf', drawstyle='steps-post')
plt.fill_between(time_points, queue_lengths, step='post', alpha=0.3, color='#17becf')

plt.axhline(max_queue_length, color='red', linestyle='--', label=f'Max Length ({max_queue_length})')

plt.title(f"ATM Vestibule Queue Length ({NUM_ATMS} Machines)", fontsize=14, fontweight='bold')
plt.xlabel("Simulation Time (Minutes)", fontsize=12)
plt.ylabel("Number of Customers in Queue", fontsize=12)
plt.legend()
plt.grid(axis='y', alpha=0.5)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.tight_layout()
plt.show()
